# Maximum altitude

## Initialization

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import numba as nb
import numpy as np
import xarray.ufuncs as xrf
import matplotlib.pyplot as plt

from tqdm.auto import trange
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets
from ray.rllib.agents import ppo

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import Environment, Stage, AP_FLIGHT_PATH_CONTROL

Instructions for updating:
non-resource variables are not supported in the long term


## Configuration

In [3]:
ray.init()

2021-05-09 01:14:30,417	INFO services.py:1267 -- View the Ray dashboard at http://127.0.0.1:8266


{'node_ip_address': '10.10.3.2',
 'raylet_ip_address': '10.10.3.2',
 'redis_address': '10.10.3.2:16244',
 'object_store_address': '/tmp/ray/session_2021-05-09_01-14-28_823968_38430/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2021-05-09_01-14-28_823968_38430/sockets/raylet',
 'webui_url': '127.0.0.1:8266',
 'session_dir': '/tmp/ray/session_2021-05-09_01-14-28_823968_38430',
 'metrics_export_port': 36238,
 'node_id': 'b7e627c8149382e85406557b305f5b0200a614eab2e878d9d8f63837'}

In [4]:
environment_configuration = {
    "dt": 0.01,
    "surface_diameter": 1737.4e3,
    "mu": 4.9048695e12,
    "stages": (
        Stage(
            dry_mass=1,
            propellant_mass=0.5,
            specific_impulse=80,
            thrust=2*1.7),
    ),
    "initial_altitude": 0,
    "initial_theta_e": radians(90),
    "initial_longitude": radians(90),
    "gamma_controller_gains": (4, 0, 0.2),
    "theta_controller_gains": (10, 0, 0.0),
    "controller_theta_dot_limits": (-1, 1),
    "autopilot_mode": AP_FLIGHT_PATH_CONTROL
}

In [5]:
trainer = ppo.PPOTrainer(env=Environment, config={
    "env_config": environment_configuration,
})

2021-05-09 01:14:33,066	INFO trainer.py:669 -- Tip: set framework=tfe or the --eager flag to enable TensorFlow eager execution
2021-05-09 01:14:33,067	INFO trainer.py:694 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
(pid=38567) WARNING:tensorflow:From /opt/conda/lib/python3.8/site-packages/tensorflow/python/compat/v2_compat.py:96: disable_resource_variables (from tensorflow.python.ops.variable_scope) is deprecated and will be removed in a future version.
(pid=38567) Instructions for updating:
(pid=38567) non-resource variables are not supported in the long term
(pid=38570) WARNING:tensorflow:From /opt/conda/lib/python3.8/site-packages/tensorflow/python/compat/v2_compat.py:96: disable_resource_variables (from tensorflow.python.ops.variable_scope) is deprecated and will be removed in a future version.
(pid=38570) Instructions for updating:
(pid=38570) non-resource variables are not supported in the long term
2021-05-09

## Training

In [12]:
training_history = {
    "episode_reward_max": [],
    "episode_reward_min": [],
    "episode_reward_mean": [],
    "episode_len_mean": []
}

out = widgets.Output()
display(out)
with out:
    display("")

for i in trange(100):
    result = trainer.train()

    out.clear_output(True)
    with out:
        display(hyr({k: result[k] for k in training_history.keys()}))

    for k in training_history.keys():
        training_history[k].append(result[k])


Output()

  0%|          | 0/100 [00:00<?, ?it/s]

2021-05-09 01:18:31,664	WARNING ppo.py:216 -- The magnitude of your environment rewards are more than 1672.0x the scale of `vf_clip_param`. This means that it will take more than 1672.0 iterations for your value function to converge. If this is not intended, consider increasing `vf_clip_param`.
2021-05-09 01:18:37,786	WARNING ppo.py:216 -- The magnitude of your environment rewards are more than 1672.0x the scale of `vf_clip_param`. This means that it will take more than 1672.0 iterations for your value function to converge. If this is not intended, consider increasing `vf_clip_param`.
2021-05-09 01:18:44,676	WARNING ppo.py:216 -- The magnitude of your environment rewards are more than 1672.0x the scale of `vf_clip_param`. This means that it will take more than 1672.0 iterations for your value function to converge. If this is not intended, consider increasing `vf_clip_param`.
2021-05-09 01:18:51,948	WARNING ppo.py:216 -- The magnitude of your environment rewards are more than 1672.0x th

KeyboardInterrupt: 

## Greedy evaluation

In [13]:
env = Environment(environment_configuration)

logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference", "vii", "xii", "fii_thrust", "mass", "mass_dot"])

In [14]:
episode_reward = 0
done = False
obs = env.reset()
while not done:
    action = trainer.compute_action(obs)
    obs, reward, done, info = env.step(action)
    episode_reward += reward
    logger.step()

episode_result = logger.episode_finish()
print(episode_reward)
episode_result

305.5172569648679


<xarray.Dataset>
Dimensions:                         (d_2_0: 2, t: 3633)
Coordinates:
  * t                               (t) float64 0.01 0.02 0.03 ... 36.32 36.33
Dimensions without coordinates: d_2_0
Data variables: (12/13)
    env_h                           (t) float64 0.0 -0.0001625 ... 177.2 177.2
    env_gamma_e                     (t) float64 -1.571 -1.571 ... -3.14 -3.14
    env_theta_e                     (t) float64 1.571 1.571 1.571 ... 2.81 2.869
    env_theta_i_dot                 (t) float64 0.0 0.0 0.0 ... -1.0 1.0 -1.0
    env_ap_comm_gamma_e             (t) float64 nan nan nan ... -1.498 -1.933
    env_ap_comm_theta_e             (t) float64 nan nan nan ... 18.21 -1.01
    ...                              ...
    env_action_autopilot_reference  (t) float64 -1.029 -0.9891 ... -1.498 -1.933
    env_vii                         (t, d_2_0) float64 -9.95e-19 ... -0.131
    env_xii                         (t, d_2_0) float64 1.064e-10 ... 1.738e+06
    env_fii_thrust                  (t, d_2_0) float64 2.082e-16 3.4 ... 0.9127
    env_mass                        (t) float64 1.5 1.5 1.5 ... 1.343 1.343
    env_mass_dot                    (t) float64 -0.004332 ... -0.004332

In [15]:
plt.figure()
smooth_signal(episode_result.env_action_autopilot_reference).plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …